# FP8 rowwise + fast accumulation, and fp32 references for all four videos: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_fp8fast_2026-11-18.md` (G1-G9).

Time: about 10 min to build SageAttention, then about 1 h 55 min of runs (~4.2 compute units). No cell waits for input. Runs are ordered by importance; the last cell downloads the zip.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit,driver_version --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)
!nvcc --version | tail -1

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10 min)

Same build and commit as before. Stops the notebook if it fails.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention checkout -q d1a57a5  # the commit measured in results/L4_sage.md
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## 33 frames, 4 prompts

Base without FP8 (~12 min), with the kernel split. G1: same hashes as the last two sessions.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/g_base_33f
!python bench/hash_frames.py results/g_base_33f --check results/f_base_33f/frames_sha256.json
!python bench/hash_frames.py results/g_base_33f --check results/pv16_33f/frames_sha256.json

FP8 rowwise with fast accumulation (~10 min). G3, G4.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --fp8-linears rowwise --fp8-fast-accum --kernels --out results/g_fp8fast_33f
!python bench/compare_frames.py --ref results/g_base_33f --test results/g_fp8fast_33f
import json
for d in ['g_base_33f', 'g_fp8fast_33f']:
    r = json.load(open(f'results/{d}/report.json'))
    f = [x['phases']['transformer']['mean_ms'] for x in r['runs']]
    print(d, 'mean forward', round(sum(f) / len(f), 1), 'ms', {k: round(v) for k, v in r['kernels']['ms'].items()})

Eager bf16 on the coast and the leaf (~8 min), for their fp32 bars (G7).

In [ ]:
!python bench/profile_baseline.py --frames 33 --prompt-ids 1 2 --out results/g_eager_33f
!python bench/hash_frames.py results/g_eager_33f --check results/flash_33f/frames_sha256.json

fp32 reference, all four prompts (~58 min), then every 33 f run against it (G6-G8).

In [ ]:
!python bench/profile_baseline.py --frames 33 --dtype fp32 --attn efficient --out results/g_fp32_33f
!python bench/hash_frames.py results/g_fp32_33f --check results/fp32_33f/frames_sha256.json
for t in ['g_base_33f', 'g_fp8fast_33f', 'g_eager_33f']:
    !python bench/compare_frames.py --ref results/g_fp32_33f --test results/{t} --out-name compare_vs_fp32.json
import json
bars = {0: 27.1, 3: 22.8}
e = {v['video']: v['psnr_mean'] for v in json.load(open('results/g_eager_33f/compare_vs_fp32.json'))['videos']}
bars.update({i: e[f'frames_{i}.npy'] - 1 for i in (1, 2)})
f = {v['video']: v['psnr_mean'] for v in json.load(open('results/g_fp8fast_33f/compare_vs_fp32.json'))['videos']}
for i in range(4):
    p = f[f'frames_{i}.npy']
    print(f'video {i}: fp8 fast vs fp32 {p:.2f} dB, bar {bars[i]:.2f} ->', 'meets' if p >= bars[i] else 'misses')

## 81 frames, fox (~19 min), G2 and G5

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/g_base_81f
!python bench/hash_frames.py results/g_base_81f --check results/f_base_81f/frames_sha256.json
!python bench/hash_frames.py results/g_base_81f --check results/pv16_81f/frames_sha256.json
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --fp8-linears rowwise --fp8-fast-accum --kernels --out results/g_fp8fast_81f
!python bench/compare_frames.py --ref results/g_base_81f --test results/g_fp8fast_81f

## Activation statistics, fox and market (~10 min), G9

Eager, no FP8; measures why one tensor scale loses accuracy.

In [ ]:
!python bench/fp8_act_stats.py --prompt-ids 0 3 --out results/g_act_stats.json

In [ ]:
# Hashes of every run; zip without .npy
import glob, os
runs = sorted(d for d in glob.glob('results/g_*') if os.path.isdir(d))
for d in runs:
    !python bench/hash_frames.py {d} > /dev/null
extra = ' '.join(p for p in ['results/g_act_stats.json'] if os.path.exists(p))
runs_str = ' '.join(runs)
!zip -qr fp8fast_results.zip {runs_str} {extra} -x '*.npy' && ls -lh fp8fast_results.zip
from google.colab import files; files.download('fp8fast_results.zip')